# SQL Data Retrieval and Validation

## Objective
Connect Python to a PostgreSQL database containing 24,386,900 credit card transactions. Retrieve selected records using SQL and validate the results against previous Pandas analysis.

## Tools
- PostgreSQL
- SQL
- Python
- Pandas
- SQLAlchemy

## Business Value
Demonstrate efficient data retrieval and validation of banking transaction data without unnecessarily loading the entire database into memory.

## 1. Environment Setup and Dependency Installation

### Objective
Prepare the Python environment with the libraries required to connect to PostgreSQL and execute SQL queries from a Jupyter Notebook.

### Required Libraries
- **SQLAlchemy:** Provides a Python interface for creating database connections and executing SQL queries.
- **Psycopg:** Serves as the PostgreSQL database driver, enabling communication between Python and PostgreSQL.
- **Pandas:** Supports data manipulation and analysis of query results.

### Implementation
The `%pip install` command installs the required packages into the active Jupyter environment.

The installation output confirms that SQLAlchemy and Psycopg are already installed, so no additional installation is required.

### Why This Matters
Establishing the required dependencies helps ensure that the analytical workflow can communicate with the database and supports reproducibility across development environments.

In [1]:
%pip install sqlalchemy "psycopg[binary]"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import sqlalchemy
import psycopg

print("Pandas:", pd.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)
print("Psycopg:", psycopg.__version__)

print("Environment ready!")

Pandas: 2.3.3
SQLAlchemy: 2.0.43
Psycopg: 3.2.9
Environment ready!


In [3]:
import os
from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Configure your own local PostgreSQL connection using environment variables.
# Never commit passwords or connection secrets to GitHub.
url = URL.create(
    drivername="postgresql+psycopg",
    username=os.getenv("PGUSER", "fraud_analyst"),
    password=os.getenv("PGPASSWORD") or getpass("PostgreSQL password: "),
    host=os.getenv("PGHOST", "localhost"),
    port=int(os.getenv("PGPORT", "5432")),
    database=os.getenv("PGDATABASE", "banking"),
)
engine = create_engine(url)
with engine.connect() as connection:
    print("Database connection successful:", connection.execute(text("SELECT 1")).scalar())

PostgreSQL password:  ········


Database connection successful: 1


## 2. Library Import and Environment Verification

### Objective
Verify that the Python environment can successfully import the libraries required for database connectivity and data analysis.

### Methodology
The following libraries are imported:

- **Pandas:** Used to organize, manipulate, and analyze tabular data.
- **SQLAlchemy:** Used to establish and manage database connections.
- **Psycopg:** Used as the PostgreSQL database adapter.

The installed version of each library is displayed to document the software environment.

### Environment Configuration

| Library | Version |
|---|---|
| Pandas | 2.3.3 |
| SQLAlchemy | 2.0.43 |
| Psycopg | 3.2.9 |

### Result
All three libraries imported successfully, and the environment returned the confirmation message `Environment ready!`.

### Interpretation
Successful imports indicate that the required libraries are available for the next stage of the analysis.

Recording library versions also improves reproducibility and makes it easier to investigate compatibility issues when the project is executed in another environment.

In [4]:
import pandas as pd

query = """
SELECT
    user_id,
    card_id,
    amount,
    use_chip,
    is_fraud
FROM fraud_transaction
WHERE is_fraud = TRUE
LIMIT 10;
"""

df_sql = pd.read_sql(query, engine)

df_sql

,user_id,card_id,amount,use_chip,is_fraud
0,0,2,$81.93,Chip Transaction,True
1,0,2,$297.86,Online Transaction,True
2,0,3,$11.04,Swipe Transaction,True
3,0,3,$68.86,Swipe Transaction,True
4,0,3,$96.19,Swipe Transaction,True
5,0,3,$298.00,Swipe Transaction,True
6,0,3,$3.72,Swipe Transaction,True
7,0,3,$75.41,Swipe Transaction,True
8,0,3,$245.41,Swipe Transaction,True
9,0,3,$196.10,Swipe Transaction,True


### Database Retrieval Validation

Successfully established a read-only connection between Python and PostgreSQL using SQLAlchemy and psycopg.

Retrieved 10 transaction records and five selected columns into a Pandas DataFrame. This demonstrates selective data retrieval without loading the entire 24,386,900-row dataset into Python memory.

**Observation:** The transaction amount is stored as text containing currency symbols and will require numeric conversion before financial analysis.

## 4. Fraud Prevalence and Class Distribution

### Business Objective

Determine the number and proportion of fraudulent transactions in the historical dataset.

Understanding fraud prevalence is essential for evaluating detection strategies, estimating potential operational risks, and selecting appropriate model evaluation metrics.

### Analytical Methodology

SQL aggregation is used to calculate:

1. The number of transactions labeled as fraudulent.
2. The total number of transactions in the dataset.
3. The percentage of transactions classified as fraudulent.

The `is_fraud` Boolean column identifies transaction status, where `TRUE` represents a fraudulent transaction and `FALSE` represents a legitimate transaction.

The SQL queries are executed against PostgreSQL, and the results are retrieved into Pandas DataFrames using `pd.read_sql()`.

### Technical Approach

- `COUNT(*)` calculates transaction counts.
- `WHERE is_fraud = TRUE` filters fraudulent records.
- `FILTER (WHERE is_fraud = TRUE)` performs conditional aggregation within the complete dataset.
- Pandas is used to calculate and display the overall fraud percentage.

In [5]:
query = """
SELECT COUNT(*) FROM fraud_transaction
WHERE is_fraud =TRUE;
"""

df_fraud_count = pd.read_sql(query, engine)

df_fraud_count

,count
0,29757


In [6]:
query = """
-- Write your SQL here
SELECT COUNT(*) AS total_transactions,
        COUNT(*) FILTER (WHERE is_fraud=TRUE) AS fraud_transactions from fraud_transaction;

"""

df_fraud_rate = pd.read_sql(query, engine)

df_fraud_rate

,total_transactions,fraud_transactions
0,24386900,29757


In [7]:
df_fraud_rate["fraud_rate"]=(100*df_fraud_rate["fraud_transactions"]/df_fraud_rate["total_transactions"])
df_fraud_rate

,total_transactions,fraud_transactions,fraud_rate
0,24386900,29757,0.12202


### Finding 1: Overall Fraud Prevalence

The dataset contains **24,386,900 credit card transactions**, of which **29,757 are labeled fraudulent**, resulting in an overall fraud rate of approximately **0.122%**.

SQL was used to aggregate transaction counts directly in PostgreSQL. Python and Pandas were then used to calculate the fraud percentage.

**Business implication:** Fraudulent transactions represent a very small proportion of the dataset. This class imbalance must be considered when developing predictive models. Accuracy alone may be misleading, so precision, recall, and precision-recall analysis will be important evaluation measures.

In [8]:
query = """
SELECT
    use_chip,
    COUNT(*) AS fraud_count
FROM fraud_transaction
WHERE is_fraud =TRUE
GROUP BY use_chip;
"""

df_fraud_method = pd.read_sql(query, engine)

df_fraud_method

,use_chip,fraud_count
0,Chip Transaction,4836
1,Online Transaction,18349
2,Swipe Transaction,6572


## 5. Fraud Risk Analysis by Transaction Method

### Business Objective

Investigate whether observed fraud rates differ across online, chip, and swipe transactions.

Understanding these differences may help financial institutions identify transaction channels that warrant additional investigation or risk-based monitoring.

### Analytical Methodology

Transactions are grouped by their payment method using the `use_chip` variable.

For each transaction method, SQL aggregation is used to calculate:

- Total number of transactions.
- Number of transactions labeled as fraudulent.
- Percentage of transactions classified as fraudulent.

Fraud rate is calculated as:

Fraud Rate (%) = (Fraudulent Transactions / Total Transactions) × 100

### Results

| Transaction Method | Total Transactions | Fraudulent Transactions | Fraud Rate |
|---|---:|---:|---:|
| Online | 2,713,220 | 18,349 | 0.6763% |
| Chip | 6,287,598 | 4,836 | 0.0769% |
| Swipe | 15,386,082 | 6,572 | 0.0427% |

### Business Interpretation

Online transactions exhibit the highest observed fraud rate at approximately 0.6763%, compared with 0.0769% for chip transactions and 0.0427% for swipe transactions.

The online fraud rate is approximately 8.8 times the chip fraud rate and 15.8 times the swipe fraud rate.

These findings suggest that transaction method may be a useful variable for fraud risk assessment.

However, the differences are descriptive associations and do not establish that a transaction method causes fraudulent activity.

### Business Implications

A financial institution could investigate whether transaction-channel information improves fraud-screening decisions.

Potential applications include:

- Risk-based transaction monitoring.
- Channel-specific fraud analysis.
- Feature engineering for predictive models.
- Prioritization of fraud investigation resources.

### Limitations

The analysis does not control for transaction amount, merchant category, customer characteristics, transaction timing, or other potential confounding factors.

Additional analysis is required before recommending channel-specific screening policies.

In [9]:
query = """
SELECT
    use_chip,
    COUNT(*) AS total_transactions,
    COUNT(*) FILTER (WHERE is_fraud = TRUE) AS fraud_transactions
FROM fraud_transaction
GROUP BY use_chip;
"""

df_method = pd.read_sql(query, engine)
df_method

,use_chip,total_transactions,fraud_transactions
0,Chip Transaction,6287598,4836
1,Online Transaction,2713220,18349
2,Swipe Transaction,15386082,6572


In [10]:
df_method["fraud_rate"] = (
    100 * df_method["fraud_transactions"]
    / df_method["total_transactions"]
)
df_method

,use_chip,total_transactions,fraud_transactions,fraud_rate
0,Chip Transaction,6287598,4836,0.076913
1,Online Transaction,2713220,18349,0.676281
2,Swipe Transaction,15386082,6572,0.042714


In [11]:
df_method.sort_values(by="fraud_rate", ascending=False)

,use_chip,total_transactions,fraud_transactions,fraud_rate
1,Online Transaction,2713220,18349,0.676281
0,Chip Transaction,6287598,4836,0.076913
2,Swipe Transaction,15386082,6572,0.042714


## Fraud Rate Analysis by Transaction Method

### Business Objective
Determine which transaction method—Online, Chip, or Swipe—has the highest observed fraud rate.

### Methodology
PostgreSQL was used to aggregate the total number of transactions and fraudulent transactions by payment method. The summarized data was imported into Python using `pd.read_sql()`, and Pandas was used to calculate fraud rates.

Fraud Rate (%) = (Fraudulent Transactions / Total Transactions) × 100

### Results

| Transaction Method | Total Transactions | Fraudulent Transactions | Fraud Rate |
|---|---:|---:|---:|
| Online | 2,713,220 | 18,349 | 0.677% |
| Chip | 6,287,598 | 4,836 | 0.077% |
| Swipe | 15,386,082 | 6,572 | 0.043% |

### Key Findings

1. Online transactions had the highest observed fraud rate at approximately 0.677%.
2. Online transactions had approximately 8.8 times the fraud rate of chip transactions and 15.8 times the fraud rate of swipe transactions.
3. Although swipe transactions represented the largest transaction volume, they had the lowest observed fraud rate.
4. Comparing fraud rates rather than fraud counts provides a more meaningful assessment of relative risk because it accounts for differences in transaction volume.

### Business Implications

The findings suggest that online transactions warrant greater attention in fraud risk monitoring.

However, a higher observed fraud rate does not justify automatically declining all online transactions. Such a policy could reject large numbers of legitimate purchases and create unnecessary financial and customer-service costs.

Further analysis should investigate transaction amounts, merchant categories, geographic patterns, and other risk indicators before recommending specific fraud prevention strategies.

### Limitations

This analysis is descriptive and does not establish that transaction method causes fraud. The observed differences may be influenced by other transaction characteristics.

### Conclusion

Online transactions represent the highest observed fraud risk among the three payment methods in this dataset. The results provide a foundation for further fraud risk modeling and evaluation of targeted detection strategies.

In [12]:
query="""
SELECT
    is_fraud,
    COUNT(*) AS transaction_count,
    AVG(REPLACE(amount, '$', '')::NUMERIC) AS avg_amount
FROM fraud_transaction
GROUP BY is_fraud;
"""
df_amount = pd.read_sql(query,engine)
df_amount

,is_fraud,transaction_count,avg_amount
0,False,24357143,43.554648
1,True,29757,108.590874


In [13]:
# Extract average fraudulent transaction amount
fraud_avg = df_amount["avg_amount"].iloc[1]

# Extract average legitimate transaction amount
legit_avg = df_amount["avg_amount"].iloc[0]

# Calculate the ratio
ratio = fraud_avg / legit_avg

# Display the result
round(ratio,2)


2.49

## Python Analysis: Comparing Average Transaction Amounts

### Objective

Calculate how many times larger the average fraudulent transaction amount is compared with the average legitimate transaction amount.

### Methodology

Using the SQL results imported into a Pandas DataFrame (`df_amount`), Python was used to:

1. Extract the average fraudulent transaction amount using `.iloc[1]`.
2. Extract the average legitimate transaction amount using `.iloc[0]`.
3. Divide the fraudulent average by the legitimate average to calculate the ratio.

The row positions were verified against the displayed DataFrame.

### Results

| Transaction Type | Average Amount |
|---|---:|
| Fraudulent | $108.59 |
| Legitimate | $43.55 |

**Ratio = 108.59 / 43.55 ≈ 2.49**

### Business Interpretation

Fraudulent transactions have an average amount approximately **2.49 times higher** than legitimate transactions.

This suggests that transaction amount may be an important feature when identifying potentially fraudulent activity.

However, a higher average does not mean every large transaction is fraudulent. Additional analysis is required to understand the distribution of transaction amounts and the overlap between fraudulent and legitimate transactions.

### Key Takeaway

Transaction amount shows a meaningful difference between fraudulent and legitimate transactions, making it a promising variable for further fraud risk analysis.

**Next Step:** Investigate fraud rates across different transaction amount ranges to determine whether higher transaction amounts are associated with greater fraud risk.

## 7. Evaluating a $100 Fraud-Screening Threshold

### Business Objective

Evaluate the potential operational impact of a rule that flags every transaction exceeding $100 for additional fraud review.

Although previous analysis found that fraudulent transactions have a higher average monetary value than legitimate transactions, this does not necessarily mean that a fixed transaction-amount threshold will accurately distinguish fraud from legitimate activity.

The objective is to quantify how many legitimate transactions would be incorrectly flagged by this screening rule.

### Screening Rule

- Transaction amount greater than $100: Flag for review.
- Transaction amount less than or equal to $100: Do not flag.

### Analytical Methodology

A PostgreSQL query counts legitimate transactions that exceed the $100 threshold.

The query applies two conditions:

1. `is_fraud = FALSE` identifies legitimate transactions.
2. `REPLACE(amount, '$', '')::NUMERIC > 100` identifies transactions with a monetary value greater than $100.

Because transaction amounts are stored as text, the dollar sign must be removed and the remaining value converted to a numeric data type before comparison.

### Classification Interpretation

Every legitimate transaction exceeding $100 is classified as a false positive under this screening rule.

A false positive occurs when a legitimate transaction is incorrectly flagged as potentially fraudulent.

This analysis measures the number of false positives generated by the rule, which is necessary for evaluating review workload and potential customer inconvenience.

In [14]:
query = """
SELECT
    COUNT(*) AS legitimate_above_100
FROM fraud_transaction
WHERE is_fraud = FALSE
  AND REPLACE(amount, '$', '')::NUMERIC > 100;
"""

df_threshold = pd.read_sql(query, engine)

df_threshold

,legitimate_above_100
0,2659851


In [15]:
false_positive_rate = (
    df_threshold["legitimate_above_100"].iloc[0]
    / df_amount["transaction_count"].iloc[0]
) * 100

round(false_positive_rate,2)

10.92

### Results: False-Positive Analysis

| Metric | Result |
|---|---:|
| Total legitimate transactions | 24,357,143 |
| Legitimate transactions above $100 | 2,659,851 |
| False-positive rate | Approximately 10.92% |

The false-positive rate is calculated as:

FPR = FP / (FP + TN) × 100

FPR = 2,659,851 / 24,357,143 × 100

**False-Positive Rate ≈ 10.92%**

### Business Interpretation

The proposed $100 screening rule would incorrectly flag approximately 2.66 million legitimate transactions in the historical dataset.

This represents approximately 10.92% of all legitimate transactions.

If every flagged transaction required manual investigation, this rule could generate a substantial operational workload.

The result demonstrates that using transaction amount alone may produce a large number of false positives, even though fraudulent transactions have a higher average value.

### Business Implications

A financial institution would need to consider the costs associated with reviewing legitimate transactions, potential transaction delays, and customer experience.

However, the rule's effectiveness cannot yet be determined because the number of fraudulent transactions successfully detected has not been calculated.

### Limitations

This analysis evaluates only the false-positive component of the screening rule.

It does not yet measure fraud recall, precision, review costs, or prevented fraud losses.

The historical false-positive rate does not establish how the rule would perform on future transactions.

### Next Step

Calculate the number of fraudulent transactions exceeding $100 to determine the rule's true positives and recall.

This will allow a more complete evaluation of the trade-off between fraud detection and unnecessary transaction reviews.

In [16]:
# Count fraudulent transactions above $100

query = """
SELECT
    COUNT(*) AS fraudulent_above_100
FROM fraud_transaction
WHERE is_fraud = TRUE
  AND REPLACE(amount, '$', '')::NUMERIC > 100;
"""

# Execute SQL query and load results into Pandas
df_true_positive = pd.read_sql(query, engine)

# Display results
df_true_positive

,fraudulent_above_100
0,11462


In [17]:
RECALL = (df_true_positive["fraudulent_above_100"].iloc[0] / df_amount["transaction_count"].iloc[1]) * 100

print(f"Recall: {RECALL:.2f}%")

Recall: 38.52%


## 8. True-Positive Identification and Fraud Recall

### Business Objective

Evaluate the effectiveness of a simple transaction screening rule that flags transactions exceeding $100 for potential fraud.

The primary objective is to determine how many actual fraudulent transactions this rule successfully identifies and what proportion of total fraudulent transactions it captures.

This analysis establishes a baseline for evaluating fraud detection performance before considering more sophisticated screening strategies or machine learning models.

### Analytical Methodology

A SQL query was executed against the `fraud_transaction` table to count transactions satisfying two conditions:

1. The transaction was confirmed fraudulent (`is_fraud = TRUE`).
2. The transaction amount exceeded $100.

Because the `amount` column was stored as text containing a dollar sign, the dollar sign was removed using `REPLACE()`, and the resulting value was converted to `NUMERIC` before comparison.

The SQL result was imported into a Pandas DataFrame. Python was then used to calculate recall by dividing the number of correctly identified fraudulent transactions by the total number of actual fraudulent transactions.

The recall formula is:

**Recall = (True Positives / Total Actual Fraudulent Transactions) × 100**

### Results

| Metric | Result |
|---|---:|
| Total fraudulent transactions | 29,757 |
| True positives (fraudulent transactions above $100) | 11,462 |
| False negatives (fraudulent transactions not flagged) | 18,295 |
| Recall | 38.52% |
| Percentage of fraud missed | 61.48% |

The calculation was:

**Recall = (11,462 / 29,757) × 100 = 38.52%**

### Business Interpretation

The $100 transaction screening rule successfully identifies approximately **38.52% of all fraudulent transactions**.

However, the rule fails to detect approximately **61.48% of fraudulent transactions**, representing 18,295 missed fraud cases.

From a banking risk management perspective, this is a significant limitation. Fraudulent transactions below or equal to $100 are not flagged by this rule, regardless of their actual fraud status.

The results demonstrate that transaction amount alone is not a sufficiently comprehensive indicator of fraudulent activity.

Although a higher transaction amount may be associated with increased fraud risk, using a single monetary threshold creates substantial gaps

In [18]:
precision = (
    df_true_positive["fraudulent_above_100"].iloc[0] /
    (
        df_threshold["legitimate_above_100"].iloc[0]
        + df_true_positive["fraudulent_above_100"].iloc[0]
    )
) * 100

print(f"Precision: {precision:.2f}%")

Precision: 0.43%
